In [ ]:
import pandas as pd
import numpy as np

from snp_analysis_tools_sherlock import *

import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
species_dir = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/merge_bacteroides/snps/102478/'
species = '102478'
info,depth,freq = load_and_sort_files(species_dir,species)
e003_coal_metadata = pd.read_csv('metadata_exp3_coal.csv').drop(columns = "Unnamed: 0")
e003_coal_metadata['mesocosm'] = e003_coal_metadata['comm'] +'-'+ e003_coal_metadata['parent_subjects'] +'-'+  e003_coal_metadata['parent_media']+'-'+ e003_coal_metadata['media']
e003_coal_metadata = e003_coal_metadata.loc[e003_coal_metadata['passage'] != 3,:]
freq = freq[np.intersect1d(freq.columns.values, e003_coal_metadata['sample'].unique())]
depth = depth[np.intersect1d(freq.columns.values, e003_coal_metadata['sample'].unique())]

In [ ]:
!ls /Users/sophiewalton/git/coalescence-pilot-mgx/workflow/out/midas2_output/merge_bacteroides/snps/102478

In [ ]:
med_nonzero_depth = depth.copy().replace(0, np.nan).median(skipna=True)
good_samples_OG = med_nonzero_depth[med_nonzero_depth>0.]
depth = depth[good_samples.index.values]
freq = freq[good_samples.index.values]

In [ ]:
good_samples_OG.sort_values()

In [ ]:
depth= depth_filtering(depth)

freq= freq_masked(freq, depth)

In [ ]:
_,diversity_df2 = get_diversity_series(freq, thresh=.2)
_,diversity_df1 = get_diversity_series(freq, thresh=.1)
diversity_df1 = pd.DataFrame(diversity_df1).reset_index().rename(columns = {0: 'diversity', 'index': 'sample'})
diversity_df2 = pd.DataFrame(diversity_df2).reset_index().rename(columns = {0: 'diversity', 'index': 'sample'})


In [ ]:
def get_parent_subjects(x):
    return e003_coal_metadata.loc[e003_coal_metadata['sample'] == x, 'parent_subjects'].values[0]

def same_parent_subjects(x):
    return x.split('-')[0]  == x.split('-')[1] 
    

In [ ]:
diversity_df1['parent_subjects'] = diversity_df1['sample'].transform(get_parent_subjects)
diversity_df1['only one subject'] = diversity_df1['parent_subjects'].transform(same_parent_subjects)
diversity_df2['parent_subjects'] = diversity_df2['sample'].transform(get_parent_subjects)
diversity_df2['only one subject'] = diversity_df2['parent_subjects'].transform(same_parent_subjects)

diversity_df1['is_inoculumn'] = diversity_df1['sample'].transform(lambda x: 'noculumn' in x)
diversity_df2['is_inoculumn'] = diversity_df2['sample'].transform(lambda x: 'noculumn' in x)
diversity_df2
p = iqplot.strip(diversity_df2, q = 'diversity',q_axis = 'y', cats = ['is_inoculumn'], 
                 color_column='only one subject', 
                 jitter=True, marker_kwargs=dict(size = 5))
p.xaxis.axis_label = 'Is an inoculumn'
bokeh.io.show(p)

In [ ]:
diversity_df2.loc[diversity_df2['diversity'] > .005,:].sort_values(by='is_inoculumn')

In [ ]:

p = iqplot.strip(diversity_df1, q = 'diversity',q_axis = 'y', cats = ['is_inoculumn'], 
                 color_column='only one subject',
                 jitter=True, marker_kwargs=dict(size = 5))
bokeh.io.show(p)

In [ ]:
samples = freq.columns.values
good_samples = []
for sample in samples:
    if 'noculumn' in sample:
        good_samples.append(sample)

In [ ]:
samples = e003_coal_metadata['sample'].values
culture_samples = []
for sample in e003_coal_metadata['sample'].values:
    if 'noculumn' not in sample:
        culture_samples.append(sample)
    


In [ ]:
samples = e003_coal_metadata.loc[e003_coal_metadata['parent_subjects'].isin(['AE-AA', 'AA-AE', 'AA-AA', 'AA-AE']),:]
samples = e003_coal_metadata.loc[e003_coal_metadata['mesocosm']== 'A10-AA-AE-mGAM-mGAM','sample'].values
samples

In [ ]:
import itertools 
good_samples = np.intersect1d(freq.columns.values, e003_coal_metadata.loc[e003_coal_metadata['is_inoculumn'],'sample'])
#dfs = []

for c1 in good_samples:
    freq_polarized = freq.loc[~freq[c1].isna(),:]
    switch = freq_polarized.loc[freq_polarized[c1] > .5].index.values
    #freq_polarized = good_site.copy()
    freq_polarized.loc[switch, :] = 1- freq.loc[switch, :]
    freq_polarized = freq_polarized.loc[freq_polarized[c1] < .2,:]
    freq_polarized[freq_polarized.isna()] = .5
    freq_polarized = freq_polarized.mask(freq_polarized < .8)
    
    
    diffs = pd.DataFrame(freq_polarized.mask(freq_polarized < .8).count()).reset_index().rename(columns = {0: 'fixed diffs', 'index': 'sample 2'})
    diffs['sample 1'] = c1
    dfs.append(diffs)
    


In [ ]:
#full_df = pd.concat(dfs)
#full_df.to_csv('diffs_lower_depth.csv')
full_df = pd.read_csv('diffs_lower_depth.csv').drop(columns = 'Unnamed: 0')
full_df['mesocosm 1'] = full_df['sample 1'].transform(lambda x: e003_coal_metadata.loc[e003_coal_metadata['sample'] == x,'mesocosm'].values[0])
full_df['mesocosm 2'] = full_df['sample 2'].transform(lambda x: e003_coal_metadata.loc[e003_coal_metadata['sample'] == x,'mesocosm'].values[0])

In [ ]:
diversity_df2_qp_samples = diversity_df2.loc[diversity_df2['diversity'] < 1e-3, 'sample'].values
med_nonzero_depth = depth.copy().replace(0, np.nan).median(skipna=True)
good_samples = med_nonzero_depth[med_nonzero_depth>5.].index.values
good_samples  = np.intersect1d(good_samples, diversity_df2_qp_samples )
full_df['s2 is inoculumn'] = full_df['sample 2'].transform(lambda x: 'noculumn' in x)
full_df_qp_refs = full_df.loc[full_df['sample 1'].isin(good_samples),:]
full_df_qp = full_df_qp_refs.copy() # .loc[full_df_qp_refs['sample 2'].isin(diversity_df2_qp_samples ),:]
med_nonzero_depth

In [ ]:
#mesocosms missing samples 


In [ ]:
missing_mesocosm = []
small_df = full_df_qp.loc[~full_df_qp['s2 is inoculumn'],:]
for m in e003_coal_metadata['mesocosm'].unique():
    if m not in small_df['mesocosm 2'].unique():
        missing_mesocosm.append(m)
len(missing_mesocosm)      

In [ ]:
len(small_df['mesocosm 2'].unique())

In [ ]:
df_full_cultures = full_df_qp.loc[~full_df_qp['s2 is inoculumn'],:]
df_full_cultures['parent subjects 1'] = df_full_cultures['mesocosm 1'].transform(lambda x: x.split('-')[1] + '-'  + x.split('-')[2])
df_full_cultures['parent subjects 2'] = df_full_cultures['mesocosm 2'].transform(lambda x: x.split('-')[1] + '-'  + x.split('-')[2])
b = df_full_cultures.loc[df_full_cultures['fixed diffs'] <100,:].sort_values(by='fixed diffs')
c = b.loc[b['parent subjects 1'].isin(['AA-AA','AF-AF','AE-AE', 'AC/PP-AC/PP']),:]
len(b['mesocosm 2'].unique())
good_samples = b.loc[~b['mesocosm 2'].isin(c['mesocosm 2'].unique()),'mesocosm 2']

In [ ]:
len(c['mesocosm 2'].unique())

In [ ]:
c

In [ ]:
diversity_df2_low = diversity_df2.loc[diversity_df2['diversity'] > 1e-3,:]
diversity_df2_low 

In [ ]:
len(df_full_cultures['sample 2'].unique())

In [ ]:
len(b['sample 2'].unique())

In [ ]:
df_full_cultures['

In [ ]:
df_full_cultures['parent subjects 1'].unique()

In [ ]:
full_df.loc[full_df['sample 1']  == 'C4-e003Coalescence-Inoculumn-mGAM',:]

In [ ]:
samples = freq.columns

In [ ]:
good_samples = []
for s in samples:
    if 'C4' in s:
        good_samples.append(s)

In [ ]:
good_samples